# Phase 1: fine-tune the pretrained MatchboxNet on our 12-class dataset

Continues `docs/plans/audio_eval_notebook_refactor_plan.md`, "Pretrained Backbone / Transfer Learning Track". Phase 0 (`colab_nemo_phase0_verification.ipynb`) already confirmed the checkpoint loads and exports to ONNX faithfully. The first run of this notebook hit 95.75% offline accuracy and 9/11 on the live-stream test -- the best result anywhere in this plan, well past the 6/11 ceiling every custom-CNN checkpoint (v3-v7) hit. This run had no explicit seed, so a real open question remains: is that representative, or this run's luck (live-stream swung 4-6/11 purely from seed in the custom-CNN track on an identical recipe)? `SEED` below makes that checkable.

Confirmed via NVIDIA's docs / live diagnostics before writing/fixing each piece (not assumed):
- Manifest format is one JSON object per line: `{"audio_filepath": ..., "duration": ..., "label": ...}`
- `model.change_labels(new_labels)` swaps the decoder for a new label set while keeping the pretrained encoder + preprocessor untouched
- This NeMo version's `EncDecClassificationModel` inherits `setup_training_data`/`setup_validation_data` from `EncDecSpeakerLabelModel` without overriding them, so the keyword args are `train_data_layer_config`/`val_data_layer_config`, not `train_data_config`/`val_data_config`
- NeMo's current models subclass `lightning.pytorch.LightningModule` (the renamed/unified package), not the legacy standalone `pytorch_lightning` -- `trainer.fit()` fails an isinstance check otherwise
- The early-stopping/checkpoint metric is `val_acc_micro_top_1` (micro = correct/total, matching every v3-v7 accuracy number in this plan), not `val_acc`
- `model.labels` is `None` after `restore_from()` on a saved checkpoint -- only populated as a side effect of `setup_training_data()`. The real label order lives in `model.cfg.labels`

**Before running:** Runtime -> Change runtime type -> GPU. Needs the *current* data package (`prepare_colab_package.py` re-run since the real `jack` recordings were added, and again since `master_evaluation_audio.wav` + the live-stream eval scripts were added for the sections below).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU -- Runtime > Change runtime type > GPU, then re-run this cell.")

In [ ]:
# Set explicitly before anything else -- the original fine-tuning run had no
# seed set at all, so we don't actually know if 95.75% offline / 9/11
# live-stream was representative or that run's luck. Change this for each
# additional run; checkpoint/report paths below are all seed-suffixed so
# multiple runs' artifacts don't collide or overwrite each other on Drive.
SEED = 0

# Which pretrained MatchboxNet checkpoint to fine-tune. 3x1x64 is the
# variant every seed0-3 result in the plan so far used (~93K params,
# confirmed via Phase 0). 3x2x64 is NVIDIA's next size step up in the same
# NGC family -- same from_pretrained()/change_labels() path, just a
# different model_name -- but its exact param count/pretrained accuracy
# have NOT been independently confirmed the way Phase 0 confirmed 3x1x64's;
# don't trust a remembered number for it, read model.cfg.labels and
# sum(p.numel() for p in model.parameters()) off the actually-loaded model
# in the next cell instead.
#
# Worth trying because our custom 13.5K-param CNN's capacity was the
# confirmed ceiling in the earlier (non-pretrained) track -- if 3x1x64
# (~93K) still isn't enough headroom for the go_green/backward weaknesses
# 4/4 seeds confirmed as repeatable, this is the next size step to test,
# not a re-run of the same experiment.
#
# Per model-iteration-constraints ("version, don't overwrite" + "isolate
# the variable you're testing"): switching this is a structural change, so
# every path below is variant-suffixed too, and the dataset/manifest/labels
# code is completely unchanged between variants -- the only thing that
# differs between a 3x1x64 run and a 3x2x64 run is this one line.
MODEL_VARIANT = "3x1x64"  # or "3x2x64"
PRETRAINED_MODEL_NAME = {
    "3x1x64": "commandrecognition_en_matchboxnet3x1x64_v2",
    "3x2x64": "commandrecognition_en_matchboxnet3x2x64_v2",
}[MODEL_VARIANT]

In [ ]:
!pip install -q "nemo_toolkit[asr]"
!pip install -q onnx onnxruntime

## Get the code + dataset

Same package format as the augmentation sweep (`prepare_colab_package.py` output) -- **must be freshly rebuilt** to include the real `jack` recordings, `master_evaluation_audio.wav`, and the live-stream eval scripts (`live_stream_eval_common.py`, `nemo_live_receiver.py`) added since earlier runs -- the original sweep's zip has none of these.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Update this to wherever you uploaded the (freshly rebuilt) zip in Drive.
DRIVE_ZIP_PATH = "/content/drive/MyDrive/ml_audio_colab_package.zip"

# Fine-tuning checkpoints land here as training progresses (via a Lightning
# ModelCheckpoint callback pointed at this Drive path below) -- survives a
# disconnect/crash the same way the augmentation sweep's Drive-persisted
# progress did, just using Lightning's own checkpointing instead of a
# hand-rolled callback since training here goes through trainer.fit(),
# not our own epoch loop. Variant+seed-specific subfolder so re-running
# with a different SEED or MODEL_VARIANT doesn't collide with or overwrite
# a previous run (model-iteration-constraints: version, don't overwrite).
import os
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ml_audio_nemo_finetune_output"
CHECKPOINT_DIR = os.path.join(DRIVE_OUTPUT_DIR, f"checkpoints_{MODEL_VARIANT}_seed{SEED}")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Output directory:", DRIVE_OUTPUT_DIR)
print("Checkpoint directory (this run):", CHECKPOINT_DIR)

In [ ]:
import zipfile
import sys

os.makedirs("host_software", exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
    zf.extractall("host_software")

HOST_SOFTWARE_DIR = os.path.abspath("host_software")
if HOST_SOFTWARE_DIR not in sys.path:
    sys.path.insert(0, HOST_SOFTWARE_DIR)

print(os.listdir(os.path.join(HOST_SOFTWARE_DIR, "ml_audio")))

In [ ]:
import nemo.collections.asr as nemo_asr
# NOT pytorch_lightning -- Lightning AI renamed/unified that legacy package
# into `lightning`, and NeMo's current models subclass
# lightning.pytorch.LightningModule. The two packages define similarly-named
# classes that are NOT the same object, so a Trainer built from the legacy
# package fails an isinstance check against a NeMo model at trainer.fit()
# time (confirmed by hitting exactly that TypeError first).
import lightning.pytorch as pl
from lightning.pytorch import seed_everything

from ml_audio.training.nemo_manifest import build_manifest_entries, write_manifest
from ml_audio.training.train_audio_command_classifier import DEFAULT_DATASET_ROOT, discover_labels

seed_everything(SEED)
print(f"Seeded with SEED={SEED}")

print("Dataset root:", DEFAULT_DATASET_ROOT)
print("Exists:", os.path.isdir(DEFAULT_DATASET_ROOT))
labels = discover_labels(DEFAULT_DATASET_ROOT)
print(f"{len(labels)} classes (alphabetical, matches labels.json convention used everywhere else in this pipeline):", labels)

## Build manifests

Reuses `nemo_manifest.py` unchanged from the local test -- only the paths differ (Colab's extracted location vs. the local dev machine), which is exactly why manifest generation happens here rather than being baked into the zip: `audio_filepath` needs to be correct for wherever the dataset actually landed.

In [ ]:
MANIFEST_DIR = "nemo_manifests"
manifest_paths = {}
for split in ("train", "val"):
    entries = build_manifest_entries(DEFAULT_DATASET_ROOT, split, labels)
    out_path = os.path.join(MANIFEST_DIR, f"{split}_manifest.json")
    write_manifest(entries, out_path)
    manifest_paths[split] = os.path.abspath(out_path)
    durations = [e["duration"] for e in entries]
    print(f"{split}: {len(entries)} entries -> {manifest_paths[split]} "
          f"(duration range {min(durations):.2f}-{max(durations):.2f}s)")

## Load the pretrained checkpoint and swap the classification head

`change_labels` reinitializes only the decoder (48 -> 12-way in effect, matching our label count) -- the pretrained encoder and preprocessor (the actual transfer-learned knowledge) stay untouched. Same checkpoint Phase 0 already verified loads and exports cleanly.

In [ ]:
model = nemo_asr.models.EncDecClassificationModel.from_pretrained(
    model_name=PRETRAINED_MODEL_NAME
)
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {PRETRAINED_MODEL_NAME}: {n_params:,} parameters")
print(f"Pretrained on {len(model.cfg.labels)} classes: {model.cfg.labels}")

model.change_labels(labels)
print(f"Decoder swapped for our {len(model.cfg.labels)} classes: {model.cfg.labels}")
assert list(model.cfg.labels) == labels, "label order mismatch after change_labels -- would silently scramble predictions"

## Wire up our data and fine-tune

Start from the pretrained model's own known-good `train_ds`/`validation_ds` config (inherited from the Speech Commands v2 training recipe) and override only what has to change for our task (manifest paths, label list, batch size) -- deliberately not reinventing preprocessor/augmentation hyperparameters that already work for this exact architecture.

Fine-tuning, not training from scratch: lower LR than the pretrained recipe likely used, modest epoch budget with early stopping on validation accuracy (matching the discipline the custom-CNN track already established -- v4/v5/v6/v7 all showed this architecture family overfits fast without it).

In [ ]:
BATCH_SIZE = 128
MAX_EPOCHS = 100

model.cfg.train_ds.manifest_filepath = manifest_paths["train"]
model.cfg.train_ds.labels = labels
model.cfg.train_ds.batch_size = BATCH_SIZE
model.cfg.train_ds.shuffle = True

model.cfg.validation_ds.manifest_filepath = manifest_paths["val"]
model.cfg.validation_ds.labels = labels
model.cfg.validation_ds.batch_size = BATCH_SIZE
model.cfg.validation_ds.shuffle = False

# Confirmed via a live diagnostic against the actual loaded model (not
# assumed from docs): this NeMo version's EncDecClassificationModel
# inherits its data-setup methods from EncDecSpeakerLabelModel without
# overriding them, so the keyword args are train_data_layer_config /
# val_data_layer_config, not train_data_config / val_data_config. That
# inherited setup_training_data also re-derives self.labels from the
# manifest's own label set (sorted, same convention discover_labels()
# already uses) -- asserted below rather than assumed, since it silently
# overwrites whatever change_labels() set two cells up.
model.setup_training_data(train_data_layer_config=model.cfg.train_ds)
model.setup_validation_data(val_data_layer_config=model.cfg.validation_ds)
assert list(model.labels) == labels, f"label order drifted after setup_training_data: {model.labels}"
print("model.labels confirmed matching our alphabetical label list after setup.")

# Lower LR than a from-scratch run -- fine-tuning an already-good encoder,
# not learning acoustic features from nothing.
model.cfg.optim.lr = 0.001
model.setup_optimization(optim_config=model.cfg.optim)

print("Data + optimizer configured.")

In [ ]:
# "val_acc" doesn't exist as a logged metric in this NeMo/Lightning version
# -- confirmed from the actual error message rather than guessed. Available:
# loss, learning_rate, global_step, training_batch_accuracy_top_0, val_loss,
# val_acc_micro_top_1, val_acc_macro. Using val_acc_micro_top_1 (micro =
# correct/total across all samples) to match the accuracy definition used
# everywhere else in this plan (every v3-v7 confusion-matrix accuracy number
# is trace(matrix)/total, i.e. micro) -- val_acc_macro would instead weight
# every class equally regardless of sample count, not comparable to those.
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename=f"matchboxnet_{MODEL_VARIANT}_finetuned-seed{SEED}-{{epoch:03d}}-{{val_acc_micro_top_1:.4f}}",
    monitor="val_acc_micro_top_1",
    mode="max",
    save_top_k=1,
)
early_stop_callback = pl.callbacks.EarlyStopping(
    monitor="val_acc_micro_top_1", mode="max", patience=15,
)

trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    max_epochs=MAX_EPOCHS,
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=False,
    enable_progress_bar=True,
)

trainer.fit(model)

print("Best checkpoint:", checkpoint_callback.best_model_path)
print("Best val_acc_micro_top_1:", checkpoint_callback.best_model_score)

## Evaluate offline -- same confusion-matrix convention as the custom-CNN track

NeMo reports its own aggregate val accuracy during training (above), but this plan has repeatedly needed the full confusion matrix to catch class-specific problems (the go_red/hold confusion wouldn't have shown up in a single aggregate accuracy number). Building it directly rather than trusting a single scalar, same as every other checkpoint evaluated in this plan.

In [ ]:
import json
import numpy as np
import soundfile as sf

best_model = nemo_asr.models.EncDecClassificationModel.load_from_checkpoint(checkpoint_callback.best_model_path)
best_model.eval()
device = next(best_model.parameters()).device

label_to_idx = {l: i for i, l in enumerate(labels)}
val_entries = [json.loads(line) for line in open(manifest_paths["val"])]

matrix = np.zeros((len(labels), len(labels)), dtype=np.int64)
with torch.no_grad():
    for entry in val_entries:
        audio, sr = sf.read(entry["audio_filepath"], dtype="float32")
        assert sr == 16000
        audio_t = torch.as_tensor(audio, device=device).unsqueeze(0)
        len_t = torch.tensor([len(audio)], device=device)
        logits = best_model(input_signal=audio_t, input_signal_length=len_t)
        pred_idx = int(logits.argmax(dim=-1).item())
        true_idx = label_to_idx[entry["label"]]
        matrix[true_idx, pred_idx] += 1

correct = int(np.trace(matrix))
total = len(val_entries)
print(f"Offline accuracy: {correct/total:.3%} ({correct}/{total})")

print(f"\n{'label':14s} recall")
for i, label in enumerate(labels):
    row_total = matrix[i].sum()
    recall = matrix[i, i] / row_total if row_total else 0.0
    print(f"{label:14s} {matrix[i, i]}/{row_total} = {recall:.1%}")

In [ ]:
from datetime import datetime, timezone

os.makedirs("reports", exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report_path = f"reports/nemo_finetune_confusion_matrix_{MODEL_VARIANT}_seed{SEED}_{timestamp}.json"
with open(report_path, "w") as f:
    json.dump({
        "generated_at": timestamp,
        "model": f"{PRETRAINED_MODEL_NAME} (fine-tuned)",
        "model_variant": MODEL_VARIANT,
        "n_params": n_params,
        "seed": SEED,
        "checkpoint": checkpoint_callback.best_model_path,
        "labels": labels,
        "accuracy": correct / total,
        "total_clips": total,
        "correct": correct,
        "matrix": matrix.tolist(),
    }, f, indent=2)
print("Saved:", report_path)

import shutil
shutil.copy(report_path, os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(report_path)))
print("Also copied to Drive:", DRIVE_OUTPUT_DIR)

## Export

`.nemo` checkpoint (for reloading/further fine-tuning later) and ONNX (for the eventual Jetson path, per Phase 0's verified export procedure). Both saved to Drive -- the per-epoch `.ckpt` files from the ModelCheckpoint callback above already are, but the packaged `.nemo`/`.onnx` exports are the actually-useful artifacts to carry forward. Filenames are seed-suffixed so downloading multiple seeds' results locally doesn't collide.

In [ ]:
nemo_path = f"matchboxnet_{MODEL_VARIANT}_finetuned_seed{SEED}.nemo"
onnx_path = f"matchboxnet_{MODEL_VARIANT}_finetuned_seed{SEED}.onnx"

best_model.save_to(nemo_path)
best_model.export(onnx_path)

shutil.copy(nemo_path, os.path.join(DRIVE_OUTPUT_DIR, nemo_path))
shutil.copy(onnx_path, os.path.join(DRIVE_OUTPUT_DIR, onnx_path))
print("Saved .nemo and .onnx checkpoints to", DRIVE_OUTPUT_DIR)

from google.colab import files
files.download(nemo_path)
files.download(onnx_path)
files.download(report_path)

## Live-stream evaluation -- the test that actually decides anything

Every offline number in this notebook, including the 95%+ accuracy above, has repeatedly turned out not to predict live continuous-stream performance across this whole plan (v5, v6, v7 all diverged from their offline results). This is the equivalent of `evaluate_live_receiver_stream.py`, driving the same rolling-buffer receiver harness (same window size, step interval, confidence/margin gating) through the same `master_evaluation_audio.wav` stream every custom-CNN checkpoint (v3-v7) was scored against, via `NemoAudioCommandReceiver` -- a separate receiver class mirroring `AudioCommandReceiver`'s structure exactly, since NeMo's model expects its own MFCC preprocessing rather than the custom CNN's spectrogram pipeline. Scored by the exact same shared logic (`live_stream_eval_common.py`) as the custom-CNN track, so the numbers are directly comparable, not just similarly-shaped.

Takes ~2 minutes wall-clock (simulates real-time cadence on purpose). Can also be run locally now -- NeMo installs cleanly on Windows after all (see plan doc), this cell is only in Colab for convenience right after training in the same session.

In [ ]:
from ml_audio.evaluations.live_stream_eval_common import run_live_stream_eval
from ml_audio.evaluations.nemo_live_receiver import NemoAudioCommandReceiver

# DEFAULT_DATASET_ROOT = .../ml_audio/data/synthetic+real_dataset_large/training_v2
# -- 3 path components below ml_audio/, so 2 levels of ".." lands at
# ml_audio/data/, not 3 (that would overshoot to ml_audio/ itself).
ML_AUDIO_DATA_DIR = os.path.normpath(os.path.join(DEFAULT_DATASET_ROOT, "..", ".."))
STREAM_PATH = os.path.join(ML_AUDIO_DATA_DIR, "02_silver", "master_evaluation_audio.wav")
assert os.path.exists(STREAM_PATH), f"missing {STREAM_PATH} -- was the data package rebuilt with EXTRA_DATA_FILES?"

live_receiver = NemoAudioCommandReceiver(nemo_path, source_file=STREAM_PATH)
live_report = run_live_stream_eval(
    live_receiver,
    model_label=f"nemo_matchboxnet_{MODEL_VARIANT} seed{SEED} (fine-tuned)",
    stream_label="data/02_silver/master_evaluation_audio.wav",
    out_dir="reports",
)

shutil.copy(live_report["_report_path"], os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(live_report["_report_path"])))
print("Live-stream report also copied to Drive:", DRIVE_OUTPUT_DIR)

## Not yet done

- **3x1x64 needs a 5th seed** before the go_green/backward weakness ranking counts as confirmed per `model-iteration-constraints` ("never rank from a small seed count" -- this project has twice seen a 3-seed ranking overturned at 5 seeds). 4/4 is suggestive, not yet sufficient on its own.
- **3x2x64 needs its own >=5-seed run** (same manifests/labels/live-stream harness, only `MODEL_VARIANT` differs) before it can be fairly compared to 3x1x64 -- comparing a single 3x2x64 run against 3x1x64's 4-seed average would be exactly the kind of small-sample ranking the same constraint warns against.
- TensorRT engine build + real Jetson AGX Orin 64GB (p3730) latency/power measurement -- Phase 0 only verified ONNX export/PyTorch-ONNX equivalence, not on-device performance. Note: earlier planning in this track loosely assumed a Jetson Orin *Nano*-class edge budget; `CLAUDE.md`'s actual compute target for large-model deployment is the AGX Orin 64GB dev kit (Ampere, Tensor Cores), a substantially larger budget -- don't undersize a candidate against the Nano-class assumption.
- Noise-mixing our own background recordings into fine-tuning (the other candidate fix for `backward`'s noise-masking weakness, alongside/instead of a larger model) -- not yet implemented here.
- Comparison against the custom-CNN track's best checkpoint (v4, or whatever supersedes it) on accuracy, live-stream performance, and resource footprint before any deployment decision.